In [ ]:
import torch

from torch import nn
from typing import Optional

class RotaryEmbedding(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.config = config

        inv_freq = self.compute_default_rope_parameters(self.config)

        self.register_buffer("inv_freq", inv_freq, persistent=False)

    @staticmethod
    def compute_default_rope_parameters(
        config,
    ) -> tuple["torch.Tensor", float]:
        """
        Computes the inverse frequencies according to the original RoPE implementation
        Args:
            config ([`~transformers.PreTrainedConfig`]):
                The model configuration.

        Returns:
            Tuple of (`torch.Tensor`, `float`), containing the inverse frequencies for the RoPE embeddings and the
            post-processing scaling factor applied to the computed cos/sin (unused in this type of RoPE).
        """
        base = config.rope_parameters["rope_theta"]
        dim = getattr(config, "head_dim", None) or config.hidden_size // config.num_attention_heads

        # Compute the inverse frequencies
        inv_freq = 1.0 / (
            base ** (torch.arange(0, dim, 2, dtype=torch.int64) / dim)
        )
        return inv_freq

    @torch.no_grad()
    def forward(self, x, position_ids):
        inv_freq_expanded = self.inv_freq[None, :, None].float().expand(position_ids.shape[0], -1, 1)
        position_ids_expanded = position_ids[:, None, :].float()

        freqs = (inv_freq_expanded.float() @ position_ids_expanded.float()).transpose(1, 2)
        emb = torch.cat((freqs, freqs), dim=-1)
        cos = emb.cos()
        sin = emb.sin()

        return cos.to(dtype=x.dtype), sin.to(dtype=x.dtype)


def rotate_half(x):
    """Rotates half the hidden dims of the input."""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)



def apply_rotary_pos_emb(q, k, cos, sin, position_ids=None, unsqueeze_dim=1):
    """Applies Rotary Position Embedding to the query and key tensors.

    Args:
        q (`torch.Tensor`): The query tensor.
        k (`torch.Tensor`): The key tensor.
        cos (`torch.Tensor`): The cosine part of the rotary embedding.
        sin (`torch.Tensor`): The sine part of the rotary embedding.
        position_ids (`torch.Tensor`, *optional*):
            Deprecated and unused.
        unsqueeze_dim (`int`, *optional*, defaults to 1):
            The 'unsqueeze_dim' argument specifies the dimension along which to unsqueeze cos[position_ids] and
            sin[position_ids] so that they can be properly broadcasted to the dimensions of q and k. For example, note
            that cos[position_ids] and sin[position_ids] have the shape [batch_size, seq_len, head_dim]. Then, if q and
            k have the shape [batch_size, heads, seq_len, head_dim], then setting unsqueeze_dim=1 makes
            cos[position_ids] and sin[position_ids] broadcastable to the shapes of q and k. Similarly, if q and k have
            the shape [batch_size, seq_len, heads, head_dim], then set unsqueeze_dim=2.
    Returns:
        `tuple(torch.Tensor)` comprising of the query and key tensors rotated using the Rotary Position Embedding.
    """
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [ ]:
import torch
import torch.nn as nn

class RotaryEmbedding(nn.Module):
    def __init__(self, kv_channels, rotary_percent=1.0, rotary_base=10000.0, seq_len_interpolation_factor=None):
        """
        Megatron-LM 风格的 Rotary Embedding 初始化。
        
        Args:
            kv_channels (int): Key/Value 的通道数 (通常是 hidden_size // num_attention_heads)。
            rotary_percent (float): 应用 RoPE 的维度比例，默认为 1.0 (全维度)。
            rotary_base (float): RoPE 的底数 (theta)，Config 中为 1000000.0。
            seq_len_interpolation_factor (float, optional): 线性插值因子，用于长序列外推。
        """
        super().__init__()
        
        self.dim = kv_channels
        self.rotary_percent = rotary_percent
        # 如果 rotary_percent < 1.0，则仅对前 dim * percent 部分应用旋转
        self.rotary_dim = int(self.dim * rotary_percent)
        self.rotary_base = rotary_base
        self.seq_len_interpolation_factor = seq_len_interpolation_factor

        # 初始化 inverse frequencies (使用 float32 保证精度)
        # 频率公式: theta_i = 1 / (base ^ (2i / d))
        inv_freq = 1.0 / (
            self.rotary_base ** (torch.arange(0, self.rotary_dim, 2).float() / self.rotary_dim)
        )
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        # 缓存 cos 和 sin 表
        self.max_seq_len_cached = 0
        self.cos_cached = None
        self.sin_cached = None

    def forward(self, max_seq_len, offset=0):
        """
        前向传播，返回特定长度的 cos 和 sin embedding。
        如果请求的长度超过缓存，会自动重新计算缓存。
        """
        # 检查是否需要更新缓存
        if max_seq_len > self.max_seq_len_cached:
            self.max_seq_len_cached = max_seq_len
            
            # 生成位置索引 t
            t = torch.arange(self.max_seq_len_cached, device=self.inv_freq.device, dtype=self.inv_freq.dtype)
            
            # 如果配置了插值因子（用于长序列处理），缩放 t
            if self.seq_len_interpolation_factor is not None and self.seq_len_interpolation_factor > 1.0:
                t /= self.seq_len_interpolation_factor

            # 计算频率: outer product [seq_len, dim/2]
            freqs = torch.outer(t, self.inv_freq)
            
            # 拼接频率以匹配 hidden_dim: [freqs, freqs] -> [seq_len, dim]
            # Megatron 的 rotate_half 风格通常不需要在这里交错，而是直接 cat
            emb = torch.cat((freqs, freqs), dim=-1)

            # 缓存 cos 和 sin, 保持为 float32 或与模型一致的 dtype (通常为了精度保持 float32)
            self.cos_cached = emb.cos()
            self.sin_cached = emb.sin()

        # 根据 offset 返回所需的切片
        return (
            self.cos_cached[offset : offset + max_seq_len, ...],
            self.sin_cached[offset : offset + max_seq_len, ...]
        )

def rotate_half(x):
    """
    将输入 x 切分为两半并旋转：[-x2, x1]
    x shape: [..., dim]
    """
    x1, x2 = x[..., :x.shape[-1] // 2], x[..., x.shape[-1] // 2:]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(t, freqs):
    """
    应用 RoPE 到张量 t。
    
    Args:
        t (Tensor): 输入张量，典型形状为 [seq_len, batch_size, num_heads, head_dim] 或 [batch, seq, ...]
        freqs (Tuple[Tensor, Tensor]): (cos, sin) 元组，形状通常为 [seq_len, 1, 1, head_dim] 以便广播
    
    Returns:
        Tensor: 应用 RoPE 后的张量
    """
    cos, sin = freqs
    # 确保 cos/sin 形状可以广播到 t
    # 假设 t 是 [seq_len, batch, heads, dim]，cos/sin 是 [seq_len, dim]
    # 我们需要将 cos/sin 调整为 [seq_len, 1, 1, dim]
    if cos.dim() == 2:
        cos = cos.unsqueeze(1).unsqueeze(1)
        sin = sin.unsqueeze(1).unsqueeze(1)
    
    # 转换精度以避免混合精度下的下溢/上溢 (Megatron 通常在 float32 下计算 RoPE)
    t_float = t.float()
    cos = cos.to(t.device).float()
    sin = sin.to(t.device).float()
    
    # 应用公式: (x * cos) + (rotate_half(x) * sin)
    out = (t_float * cos) + (rotate_half(t_float) * sin)
    
    return out.type_as(t)

In [ ]:
# --- 实例化与使用示例 ---

# 从 config 读取参数并实例化

from utils import ModelConfig

config = ModelConfig()


# 注意：Config 中 kv_channels 通常等于 hidden_size / num_attention_heads (2048/16=128)
rope_model = RotaryEmbedding(
    kv_channels=config.kv_channels,
    rotary_base=config.rotary_base,  # Config 中为 1,000,000.0
    rotary_percent=1.0               # Megatron 默认为 1.0
)

# 2. 模拟训练过程中的一次前向传播
def training_step_example(batch_query, batch_key):
    seq_len = batch_query.shape[0] # 假设布局为 [Sequence, Batch, Heads, Dim]
    
    # 获取当前序列长度对应的 cos, sin
    cos, sin = rope_model(max_seq_len=seq_len)
    
    # 应用 RoPE
    # 注意：apply_rotary_pos_emb 会处理广播维度
    q_embed = apply_rotary_pos_emb(batch_query, (cos, sin))
    k_embed = apply_rotary_pos_emb(batch_key, (cos, sin))
    
    return q_embed, k_embed

# 创建一些伪造数据进行测试 [Seq, Batch, Heads, Dim]
dummy_q = torch.randn(config.seq_length, 2, config.num_attention_heads, config.kv_channels)
dummy_k = torch.randn(config.seq_length, 2, config.num_attention_heads, config.kv_channels)

q_out, k_out = training_step_example(dummy_q, dummy_k)

print(f"Rotary Base: {rope_model.rotary_base}")
print(f"Input shape: {dummy_q.shape}")
print(f"Output shape: {q_out.shape}")

In [ ]:
import torch

# 设置随机种子保证可复现z z z z z z za
torch.manual_seed(42)

# ==========================================
# 准备工作：生成频率 (Common)
# ==========================================
def precompute_freqs_cis(head_dim: int, seq_len: int, theta: float = 10000.0):
    # 计算角度 theta_i
    # 注意：根据 LLaMA 实现，频率只生成 dim/2 个
    # head_dim = 64 -> freqs last dim = 32
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    
    # outer product: [seq_len, head_dim/2]
    freqs = torch.outer(t, freqs)
    return freqs

# ==========================================
# 版本 A: LLaMA/Qwen 官方实数实现 (Real)
# ==========================================
def rotate_half(x):
    """(x1, x2) -> (-x2, x1)"""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope_real_llama(x, freqs):
    # x: [bs, seq, heads, dim]
    # freqs: [seq, dim/2]
    
    # 1. 为了让 cos/sin 能跟 x (dim) 对应，需要把 freqs (dim/2) 复制拼接一次
    # [seq, dim/2] -> [seq, dim]
    freqs_full = torch.cat((freqs, freqs), dim=-1)
    
    # 2. 调整形状以广播: [1, seq, 1, dim]
    freqs_full = freqs_full.view(1, x.shape[1], 1, x.shape[-1])
    
    cos = freqs_full.cos()
    sin = freqs_full.sin()
    
    # 3. 核心公式: x * cos + rotate_half(x) * sin
    return (x * cos) + (rotate_half(x) * sin)

# ==========================================
# 版本 B: 复数等价实现 (Complex - Fixed)
# ==========================================
def apply_rope_complex_llama(x, freqs):
    # x: [bs, seq, heads, dim] (e.g. dim=64)
    dim = x.shape[-1]
    
    # 1. 构造复数
    # LLaMA 的逻辑是: index i 和 index i + dim/2 是一对
    # x_real: [..., 32], x_imag: [..., 32]
    x_real = x[..., : dim // 2]
    x_imag = x[..., dim // 2 :]
    x_complex = torch.complex(x_real, x_imag) # shape: [bs, seq, heads, 32]
    
    # 2. 构造复数旋转因子
    # freqs: [seq, 32]
    # 【修复点】这里 reshape 的最后一维必须是 32 (即 dim // 2)，而不是 64
    freqs = freqs.view(1, x.shape[1], 1, dim // 2) 
    
    # 生成复数旋转子 (模长为1, 角度为 freqs)
    freqs_cis = torch.polar(torch.ones_like(freqs), freqs)
    
    # 3. 复数乘法 (旋转)
    # [bs, seq, heads, 32] * [1, seq, 1, 32]
    x_out_complex = x_complex * freqs_cis
    
    # 4. 还原回实数 (拼接实部和虚部)
    # 对应 LLaMA 的布局：先放实部(前半截)，再放虚部(后半截)
    return torch.cat([x_out_complex.real, x_out_complex.imag], dim=-1)

# ==========================================
# 比较测试
# ==========================================
def run_comparison():
    # 参数设置
    bs, seq_len, n_heads, head_dim = 2, 128, 4, 64
    
    # 随机输入
    x = torch.randn(bs, seq_len, n_heads, head_dim)
    
    # 预计算频率 (输出 shape: [128, 32])
    freqs = precompute_freqs_cis(head_dim, seq_len)
    
    # 运行两个版本
    output_real = apply_rope_real_llama(x.clone(), freqs)
    output_complex = apply_rope_complex_llama(x.clone(), freqs)
    
    # 检查误差
    is_close = torch.allclose(output_real, output_complex, atol=1e-6)
    max_diff = (output_real - output_complex).abs().max().item()
    
    print(f"输入形状: {x.shape}")
    print(f"Freqs形状: {freqs.shape}")
    print(f"版本 A (实数/Llama) 输出形状: {output_real.shape}")
    print(f"版本 B (复数/Complex) 输出形状: {output_complex.shape}")
    print("-" * 30)
    print(f"最大差异 (Max Difference): {max_diff:.8f}")
    print(f"结果是否一致 (torch.allclose): {is_close}")
    
    if is_close:
        print("\n✅ 验证成功：复数写法与 LLaMA 实数写法完全等价！")
    else:
        print("\n❌ 验证失败：结果不一致。")

if __name__ == "__main__":
    run_comparison()

In [ ]:
import torch
import torch.nn as nn

class RotaryEmbedding(nn.Module):
    def __init__(self, config):
        """
        初始化 RoPE，参数直接从 config 读取。
        """
        super().__init__()
        # 从 config 中获取关键参数
        self.config = config
        
        # 预计算逆频率 (Inverse Frequencies)
        # 频率公式: 1.0 / (base ** (i / dim))，其中 i 是 0, 2, 4...
        inv_freq = 1.0 / (self.config.rotary_base ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        # 缓存 Cos 和 Sin 表，避免每次 forward 都重新计算
        self._set_cos_sin_cache(self.config.max_position_embeddings)

    def _set_cos_sin_cache(self, seq_len, device=None, dtype=torch.get_default_dtype()):
        """预计算并缓存 cos 和 sin 表"""
        self.max_seq_len = seq_len
        t = torch.arange(self.max_seq_len, device=device, dtype=self.inv_freq.dtype)
        
        # 计算频率张量: [seq_len, dim/2]
        freqs = torch.outer(t, self.inv_freq)
        
        # 拼接成 [seq_len, dim] 的形状，使得最后一位包含 (freq, freq)
        # 这是为了配合 rotate_half 操作
        emb = torch.cat((freqs, freqs), dim=-1)
        
        # 注册为 buffer，形状通常为 [max_seq_len, 1, 1, dim] 以支持广播
        # 注意：实际形状取决于你的输入 layout (Batch, Seq, Head, Dim) 还是 (Seq, Batch, Head, Dim)
        # 这里为了通用性，我们先保持 [max_seq_len, dim]，在 forward 中调整
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, x, seq_len=None):
        """
        x: 输入张量，用于推断设备和类型
        seq_len: 当前序列长度
        """
        if seq_len > self.max_seq_len:
            self._set_cos_sin_cache(seq_len, device=x.device, dtype=x.dtype)
            
        # 获取当前长度的切片
        return (
            self.cos_cached[:seq_len, ...].to(dtype=x.dtype),
            self.sin_cached[:seq_len, ...].to(dtype=x.dtype),
        )

def rotate_half(x):
    """
    将输入张量 x 切分为两半并旋转。
    x 的形状假设为 [..., dim]
    [-x2, x1]
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    """
    应用旋转位置编码。
    x: [batch, seq_len, n_head, head_dim] 或 [seq_len, batch, n_head, head_dim]
    cos, sin: 形状应能广播到 x
    """
    # 确保 cos/sin 能够广播到 x 的维度。
    # 如果 x 是 [batch, seq_len, head, dim]，cos/sin 应该是 [1, seq_len, 1, dim]
    # 这里做一个简单的维度对齐示例，具体需根据数据格式调整
    if cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(2) # [1, seq_len, 1, dim]
        sin = sin.unsqueeze(0).unsqueeze(2)
        
    return (x * cos) + (rotate_half(x) * sin)

# --- 使用示例 ---
if __name__ == "__main__":
    # 模拟输入数据 [Batch=2, Seq=10, Heads=16, Dim=128]
    batch_size, current_seq_len = 2, 10
    q = torch.randn(batch_size, current_seq_len, config.num_attention_heads, config.kv_channels)
    k = torch.randn(batch_size, current_seq_len, config.num_attention_heads, config.kv_channels)

    # 1. 初始化 RoPE 模块
    rope = RotaryEmbedding(config)

    # 2. 获取当前序列长度的 cos 和 sin
    cos, sin = rope(q, seq_len=current_seq_len)

    # 3. 应用 RoPE
    q_rope = apply_rotary_pos_emb(q, cos, sin)
    k_rope = apply_rotary_pos_emb(k, cos, sin)

    print(f"Original Q shape: {q.shape}")
    print(f"RoPE Q shape:     {q_rope.shape}")
    print(f"Rotary Base:      {rope.base}")

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
from dataclasses import dataclass
from typing import Tuple, Optional

@dataclass
class ModelArgs:
    dim: int = 512
    n_heads: int = 8
    max_seq_len: int = 1024
    vocab_size: int = 1000

# 1. 生成 Cos 和 Sin 表 (LLaMA/Qwen 风格)
def precompute_cos_sin(dim: int, seq_len: int, theta: float = 10000.0):
    # 计算 theta_i，维度是 head_dim // 2
    inv_freq = 1.0 / (theta ** (torch.arange(0, dim, 2).float() / dim))
    t = torch.arange(seq_len, device=inv_freq.device)
    
    # 计算 outer product: [seq_len, head_dim // 2]
    freqs = torch.outer(t, inv_freq)
    
    # 关键差异：LLaMA 逻辑是将 freqs 拼接两次，以适配 rotate_half 的对半切分
    # 结果 shape: [seq_len, head_dim]
    emb = torch.cat((freqs, freqs), dim=-1)
    
    # 返回 cos 和 sin (实数)
    return emb.cos(), emb.sin()

# 2. 定义 rotate_half 操作
def rotate_half(x: torch.Tensor):
    """
    将输入张量的最后一维分成两半，并进行交换和符号变换。
    x: [..., d] -> x1: [..., d/2], x2: [..., d/2]
    out: cat(-x2, x1)
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

# 3. 旋转位置编码计算 (实数域)
def apply_rotary_emb(
    xq: torch.Tensor,
    xk: torch.Tensor,
    cos: torch.Tensor,
    sin: torch.Tensor,
) -> Tuple[torch.Tensor, torch.Tensor]:
    # xq.shape = [batch_size, seq_len, n_heads, head_dim]
    # cos, sin 原始 shape = [seq_len, head_dim]
    
    # 调整 cos, sin 形状以支持广播: [seq_len, head_dim] -> [1, seq_len, 1, head_dim]
    # 注意：这里假设 seq_len 维度已经切片匹配了
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)
    
    # 应用公式: (x * cos) + (rotate_half(x) * sin)
    xq_embed = (xq * cos) + (rotate_half(xq) * sin)
    xk_embed = (xk * cos) + (rotate_half(xk) * sin)
    
    return xq_embed.type_as(xq), xk_embed.type_as(xk)

class Attention(nn.Module):
    def __init__(self, args: ModelArgs):
        super().__init__()
        self.n_heads = args.n_heads
        self.head_dim = args.dim // args.n_heads
        
        self.wq = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wk = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wv = nn.Linear(args.dim, args.n_heads * self.head_dim, bias=False)
        self.wo = nn.Linear(args.n_heads * self.head_dim, args.dim, bias=False)
        
        # 预计算 Cos 和 Sin 并注册为 buffer
        # 这里传入 self.head_dim
        cos, sin = precompute_cos_sin(self.head_dim, args.max_seq_len * 2)
        self.register_buffer("cos_cached", cos)
        self.register_buffer("sin_cached", sin)

    def forward(self, x: torch.Tensor):
        bsz, seqlen, _ = x.shape
        
        # 1. 投影
        xq, xk, xv = self.wq(x), self.wk(x), self.wv(x)

        # 2. 拆分多头 [batch, seq, n_heads, head_dim]
        xq = xq.view(bsz, seqlen, self.n_heads, self.head_dim)
        xk = xk.view(bsz, seqlen, self.n_heads, self.head_dim)
        xv = xv.view(bsz, seqlen, self.n_heads, self.head_dim)

        # 3. 应用 RoPE (使用 LLaMA/Qwen 风格)
        # 根据当前 seqlen 切片
        cos = self.cos_cached[:seqlen] 
        sin = self.sin_cached[:seqlen]
        
        xq, xk = apply_rotary_emb(xq, xk, cos=cos, sin=sin)

        # 4. 转置以进行 Attention 计算 [batch, n_heads, seq, head_dim]
        xq = xq.transpose(1, 2)
        xk = xk.transpose(1, 2)
        xv = xv.transpose(1, 2)
        
        # 5. 计算 Attention Scores
        # scores.shape = (bsz, n_heads, seqlen, seqlen)
        scores = torch.matmul(xq, xk.transpose(-2, -1)) / math.sqrt(self.head_dim)
        scores = F.softmax(scores.float(), dim=-1).type_as(xq)
        
        # 6. 计算 Output
        output = torch.matmul(scores, xv)  # (bsz, n_heads, seqlen, head_dim)
        
        # 7. 还原形状并投影输出
        output = output.transpose(1, 2).contiguous().view(bsz, seqlen, -1)
        return self.wo(output)

# --- 测试代码 ---
if __name__ == "__main__":
    args = ModelArgs()
    model = Attention(args)
    
    # 创建模拟输入 [batch_size, seq_len, dim]
    x = torch.randn(2, 32, args.dim)
    
    try:
        y = model(x)
        print(f"输入形状: {x.shape}")
        print(f"输出形状: {y.shape}")
        print("代码运行成功！")
        
        # 打印缓存形状验证
        print(f"Cos Cache Shape: {model.cos_cached.shape}")
        
    except Exception as e:
        print(f"运行出错: {e}")
        import traceback
        traceback.print_exc()

In [ ]:
import torch

# ==========================================
# 你的版本 A (LLaMA/Qwen)
# ==========================================
def precompute_freqs_cis(head_dim: int, seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs = torch.cat((freqs, freqs), dim=-1)
    return freqs

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope_version_a(x, freqs):
    # 调整形状以广播 (假设 x 是 [bs, seq, heads, dim])
    freqs = freqs.view(1, x.shape[1], 1, x.shape[-1])
    
    cos = freqs.cos()
    sin = freqs.sin()
    return (x * cos) + (rotate_half(x) * sin)

# ==========================================
# 我之前的版本 B (Megatron Style)
# ==========================================
class RotaryEmbeddingVersionB(torch.nn.Module):
    def __init__(self, dim, max_seq_len=4096, base=10000.0):
        super().__init__()
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq)
        self.max_seq_len = max_seq_len
        self.dim = dim

    def forward(self, x, seq_len):
        t = torch.arange(seq_len, device=x.device, dtype=self.inv_freq.dtype)
        freqs = torch.outer(t, self.inv_freq)
        # 这里直接在缓存阶段做了 concat
        emb = torch.cat((freqs, freqs), dim=-1)
        # 返回 cos, sin
        return emb.cos(), emb.sin()

def apply_rope_version_b(x, cos, sin):
    # 简单的维度对齐广播
    cos = cos.unsqueeze(0).unsqueeze(2) # [1, seq, 1, dim]
    sin = sin.unsqueeze(0).unsqueeze(2)
    return (x * cos) + (rotate_half(x) * sin)

In [ ]:
# ==========================================
# 验证脚本
# ==========================================
torch.manual_seed(42)

# 参数设置
bs, seq, heads, dim = 2, 128, 4, 64
theta = 10000.0

# 输入数据
q = torch.randn(bs, seq, heads, dim)
k = torch.ran

# --- 运行版本 A ---
freqs_a = precompute_freqs_cis(dim, seq, theta)
out_a = apply_rope_version_a(x, freqs_a)

# --- 运行版本 B ---
rope_b = RotaryEmbeddingVersionB(dim, max_seq_len=seq, base=theta)
cos_b, sin_b = rope_b(x, seq_len=seq) # 获取 cos/sin
out_b = apply_rope_version_b(x, cos_b, sin_b)

# --- 比较 ---
# 使用 allclose 比较浮点数是否足够接近
is_close = torch.allclose(out_a, out_b, atol=1e-5)
max_diff = (out_a - out_b).abs().max()

print(f"最大差异: {max_diff:.8f}")
print(f"两个实现是否等价? {'✅ 是' if is_close else '❌ 否'}")

In [ ]:
import torch
import torch.nn as nn

class RotaryEmbedding(nn.Module):
    def __init__(self, config):
        """
        初始化 RoPE，参数直接从 config 读取。
        """
        super().__init__()
        self.config = config
        
        # 预计算逆频率 (Inverse Frequencies), 频率公式: 1.0 / (base ** (i / dim))，其中 i 是 0, 2, 4...
        inv_freq = 1.0 / (self.config.rotary_base ** (torch.arange(0, self.dim, 2).float() / self.dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)
        
        # 缓存 Cos 和 Sin 表，避免每次 forward 都重新计算
        self._set_cos_sin_cache(self.config.max_position_embeddings)

    def _set_cos_sin_cache(self, seq_len, device=None, dtype=torch.get_default_dtype()):
        """预计算并缓存 cos 和 sin 表"""
        self.max_seq_len = seq_len
        t = torch.arange(self.max_seq_len, device=device, dtype=self.inv_freq.dtype)
        
        # 计算频率张量: [seq_len, dim/2]
        freqs = torch.outer(t, self.inv_freq)
        
        # 拼接成 [seq_len, dim] 的形状，使得最后一位包含 (freq, freq), 为了配合 rotate_half 操作
        emb = torch.cat((freqs, freqs), dim=-1)
        
        # 注册为 buffer，形状通常为 [max_seq_len, 1, 1, dim] 以支持广播
        # 注意：实际形状取决于你的输入 layout (Batch, Seq, Head, Dim) 还是 (Seq, Batch, Head, Dim)
        # 这里为了通用性，我们先保持 [max_seq_len, dim]，在 forward 中调整
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)

    def forward(self, x):
        """
        x: 输入张量，用于推断设备和类型
        seq_len: 当前序列长度
        """
            
        # 获取当前长度的切片
        return self.cos_cached.to(dtype=x.dtype), self.sin_cached.to(dtype=x.dtype)

def rotate_half(x):
    """
    将输入张量 x 切分为两半并旋转。
    x 的形状假设为 [..., dim]
    [-x2, x1]
    """
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(x, cos, sin):
    """
    应用旋转位置编码。
    x: [batch, seq_len, n_head, head_dim] 或 [seq_len, batch, n_head, head_dim]
    cos, sin: 形状应能广播到 x
    """
    # 确保 cos/sin 能够广播到 x 的维度。
    # 如果 x 是 [batch, seq_len, head, dim]，cos/sin 应该是 [1, seq_len, 1, dim]
    # 这里做一个简单的维度对齐示例，具体需根据数据格式调整
    if cos.dim() == 2:
        cos = cos.unsqueeze(0).unsqueeze(2) # [1, seq_len, 1, dim]
        sin = sin.unsqueeze(0).unsqueeze(2)
        
    return (x * cos) + (rotate_half(x) * sin)

# --- 使用示例 ---
if __name__ == "__main__":
    # 模拟输入数据 [Batch=2, Seq=10, Heads=16, Dim=128]
    batch_size, current_seq_len = 2, 10
    q = torch.randn(batch_size, current_seq_len, config.num_attention_heads, config.kv_channels)
    k = torch.randn(batch_size, current_seq_len, config.num_attention_heads, config.kv_channels)

    # 1. 初始化 RoPE 模块
    rope = RotaryEmbedding(config)

    # 2. 获取当前序列长度的 cos 和 sin
    cos, sin = rope(q, seq_len=current_seq_len)

    # 3. 应用 RoPE
    q_rope = apply_rotary_pos_emb(q, cos, sin)
    k_rope = apply_rotary_pos_emb(k, cos, sin)

    print(f"Original Q shape: {q.shape}")
    print(f"RoPE Q shape:     {q_rope.shape}")
    print(f"Rotary Base:      {rope.base}")

In [ ]:
import torch
from torch import nn

class RotaryEmbedding(nn.Module):
    def __init__(self, dim, base=10000.0):
        super().__init__()
        # 计算 inv_freq, 频率公式: 1.0 / (base ** (i / dim))，其中 i 是 0, 2, 4...
        inv_freq = 1.0 / (base ** (torch.arange(0, dim, 2).float() / dim))
        self.register_buffer("inv_freq", inv_freq, persistent=False)

    def forward(self, x, position_ids):
        # 改用更直观的广播机制: [bs, seq, 1] * [dim/2] -> [bs, seq, dim/2]
        freqs = position_ids.unsqueeze(-1).float() * self.inv_freq
        
        # 拼接成完整的 dim: [bs, seq, dim]
        emb = torch.cat((freqs, freqs), dim=-1)
        
        # 转回输入的 dtype (如 fp16/bf16)
        return emb.cos().to(x.dtype), emb.sin().to(x.dtype)

def rotate_half(x):
    """保持原版核心逻辑: x1, x2 -> -x2, x1"""
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin, unsqueeze_dim=1):
    """
    简化 3: 移除废弃的 position_ids 参数
    
    Args:
        q, k: [batch, heads, seq, dim]
        cos, sin: [batch, seq, dim] (来自 RotaryEmbedding.forward)
        unsqueeze_dim: 用于广播的维度。如果是 [b, h, s, d] 格式，通常为 1
    """
    # [batch, seq, dim] -> [batch, 1, seq, dim]
    cos = cos.unsqueeze(unsqueeze_dim)
    sin = sin.unsqueeze(unsqueeze_dim)
    
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)
    return q_embed, k_embed

In [1]:
import torch

def compute_freqs(head_dim: int, seq_len: int, theta: float = 10000.0):
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    t = torch.arange(seq_len, dtype=torch.float32)
    freqs = torch.outer(t, freqs)
    freqs = torch.cat((freqs, freqs), dim=-1)
    return freqs

def rotate_half(x):
    x1 = x[..., : x.shape[-1] // 2]
    x2 = x[..., x.shape[-1] // 2 :]
    return torch.cat((-x2, x1), dim=-1)

def apply_rope_element(x, freqs):
    # 调整形状以广播 (假设 x 是 [bs, seq, heads, dim])
    freqs = freqs.view(1, x.shape[1], 1, x.shape[-1])
    
    cos = freqs.cos()
    sin = freqs.sin()
    return (x * cos) + (rotate_half(x) * sin)

In [2]:
import torch

def get_rope_matrix(t: int, head_dim: int, theta: float = 10000.0) -> torch.Tensor:
    """
    构建位置 t 的 RoPE 旋转矩阵 (Block-Diagonal)。
    对应 Llama/HF 的 rotate_half 策略: [x, x_half] -> [-x_half, x]
    """
    # 1. 计算频率与角度
    freqs = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = t * freqs
    cos, sin = torch.cos(angles), torch.sin(angles)
    
    # 2. 构建分块旋转矩阵
    # [ diag(cos)   diag(-sin) ]
    # [ diag(sin)   diag(cos)  ]
    
    half_dim = head_dim // 2
    rotary_matrix = torch.zeros(head_dim, head_dim)
    
    # 填充四个分块
    rotary_matrix[:half_dim, :half_dim] = torch.diag(cos)   # Top-Left
    rotary_matrix[:half_dim, half_dim:] = torch.diag(-sin)  # Top-Right (Negative)
    rotary_matrix[half_dim:, :half_dim] = torch.diag(sin)   # Bottom-Left
    rotary_matrix[half_dim:, half_dim:] = torch.diag(cos)   # Bottom-Right
    
    return rotary_matrix

def apply_rope_matrix(x: torch.Tensor, theta: float = 10000.0) -> torch.Tensor:
    """
    使用纯矩阵乘法应用 RoPE。
    Args:
        x: 输入张量 [bs, seq_len, heads, head_dim]
    """
    _, seq_len, _, head_dim = x.shape
    output = torch.zeros_like(x)
    
    for t in range(seq_len):
        # 1. 获取当前位置 t 的旋转矩阵
        rotary_matrix = get_rope_matrix(t, head_dim, theta)
        
        # 2. 提取当前时刻向量 [bs, heads, dim]
        x_t = x[:, t, :, :]
        
        # 3. 执行矩阵乘法: x_rotated = x @ rotary_matrix.T
        output[:, t, :, :] = torch.matmul(x_t, rotary_matrix.T)
        
    return output

In [5]:
import torch

# 配置
batch_size = 8
seq_len = 4096
heads = 16
head_dim = 128
theta = 10000.0

# 输入
x = torch.randn(batch_size, seq_len, heads, head_dim)

print(f"输入形状: {x.shape}")
print("正在运行对比...\n")

# --- 运行模块 1 (你的代码) ---
freqs = compute_freqs(head_dim, seq_len, theta)
out_vector = apply_rope_element(x, freqs)

# --- 运行模块 2 (矩阵实现) ---
out_matrix = apply_rope_matrix(x, theta)

# --- 验证结果 ---
# 检查最大误差
diff = (out_vector - out_matrix).abs().max()

print(f"方法 A (向量版) 样本: {out_vector[0, 1, 0, :4].tolist()}")
print(f"方法 B (矩阵版) 样本: {out_matrix[0, 1, 0, :4].tolist()}")
print("-" * 30)
print(f"最大误差: {diff.item():.9f}")

输入形状: torch.Size([8, 4096, 16, 128])
正在运行对比...

方法 A (向量版) 样本: [0.8402013778686523, -0.7141323685646057, -0.6728672981262207, -0.7416683435440063]
方法 B (矩阵版) 样本: [0.8402014374732971, -0.7141323685646057, -0.6728672385215759, -0.7416683435440063]
------------------------------
最大误差: 0.000000477


In [22]:
import torch

# 配置
batch_size = 8
seq_len = 2048
heads = 16
head_dim = 16
theta = 10000.0

# 输入
x = torch.randn(batch_size, seq_len, heads, head_dim)

print(f"输入形状: {x.shape}")
print("正在运行对比...\n")

# --- 运行模块 1 (向量版) ---
freqs = compute_freqs(head_dim, seq_len, theta)
out_vector = apply_rope_element(x, freqs)

# --- 运行模块 2 (矩阵版) ---
out_matrix = apply_rope_matrix(x, theta)

# --- 验证结果 ---
# 1. 计算差值张量
diff_tensor = (out_vector - out_matrix).abs()

# 2. 获取最大误差值
max_diff = diff_tensor.max()

# 3. 获取最大误差发生的位置（扁平化索引）
max_flat_idx = diff_tensor.argmax()

# 4. 根据索引取出两个对应的具体数值
# .view(-1) 将张量拉平，方便使用 argmax 得到的索引
val_vec = out_vector.view(-1)[max_flat_idx].item()
val_mat = out_matrix.view(-1)[max_flat_idx].item()

# 5. (可选) 获取多维索引，看看具体是哪个 Batch/Seq/Head/Dim 出的问题
# unravel_index 在 PyTorch 中可以用 nonzero 替代
coords = (diff_tensor == max_diff).nonzero(as_tuple=False)[0].tolist()

print("-" * 30)
print(f"最大误差: {max_diff.item():.9f}")
print("-" * 30)
print(f"造成最大误差的两个元素:")
print(f"向量版数值: {val_vec:.20f}")
print(f"矩阵版数值: {val_mat:.20f}")
print(f"发生位置 (Batch, Seq, Head, Dim): {coords}")
print("-" * 30)

# 验证浮点精度差异
print(f"直接相减结果: {val_vec - val_mat:.20f}")

输入形状: torch.Size([8, 2048, 16, 16])
正在运行对比...

------------------------------
最大误差: 0.000000477
------------------------------
造成最大误差的两个元素:
向量版数值: 5.13407421112060546875
矩阵版数值: 5.13407373428344726562
发生位置 (Batch, Seq, Head, Dim): [0, 126, 15, 4]
------------------------------
直接相减结果: 0.00000047683715820312


In [24]:
import torch

# 1. 强制使用 float64 (Double Precision)
dtype = torch.float64
torch.set_default_dtype(dtype) # 为了保险，设置默认浮点类型

batch_size = 8
seq_len = 128
heads = 16
head_dim = 128
theta = 10000.0

print(f"当前精度模式: {dtype}")

# 输入数据也必须是 float64
x = torch.randn(batch_size, seq_len, heads, head_dim, dtype=dtype)

# ==========================================
# 修正后的函数：严防死守，绝对不转 float32
# ==========================================

def compute_freqs_pure_64(head_dim, seq_len, theta, dtype):
    # 错误写法: .float() 会降级
    # 正确写法: 直接除，或者 .type(dtype)
    
    # 1. 生成索引 [0, 2, 4, ...] (int64)
    idx = torch.arange(0, head_dim, 2, device=x.device)
    
    # 2. 转为 float64 进行除法
    idx_float = idx.to(dtype) 
    
    freqs = 1.0 / (theta ** (idx_float / head_dim))
    
    t = torch.arange(seq_len, dtype=dtype, device=x.device)
    freqs = torch.outer(t, freqs)
    freqs = torch.cat((freqs, freqs), dim=-1)
    return freqs

def get_rope_matrix_pure_64(t, head_dim, theta, dtype):
    # 保持完全一致的计算逻辑
    idx = torch.arange(0, head_dim, 2, device=x.device)
    idx_float = idx.to(dtype)
    
    freqs = 1.0 / (theta ** (idx_float / head_dim))
    
    # 这里的 t 也是 python int，自动广播没问题
    angles = t * freqs
    cos, sin = torch.cos(angles), torch.sin(angles)
    
    half = head_dim // 2
    mat = torch.zeros((head_dim, head_dim), dtype=dtype, device=x.device)
    mat[:half, :half] = torch.diag(cos)
    mat[:half, half:] = torch.diag(-sin)
    mat[half:, :half] = torch.diag(sin)
    mat[half:, half:] = torch.diag(cos)
    return mat

# ==========================================
# 运行对比
# ==========================================

# 1. 向量版计算
freqs = compute_freqs_pure_64(head_dim, seq_len, theta, dtype)
freqs_reshaped = freqs.view(1, seq_len, 1, head_dim)

cos = freqs_reshaped.cos()
sin = freqs_reshaped.sin()

x1 = x[..., :head_dim//2]
x2 = x[..., head_dim//2:]
rotate_x = torch.cat((-x2, x1), dim=-1)
out_vector = (x * cos) + (rotate_x * sin)

# 2. 矩阵版计算
out_matrix = torch.zeros_like(x)
for t in range(seq_len):
    mat = get_rope_matrix_pure_64(t, head_dim, theta, dtype)
    # 显式转置，确保维度匹配
    out_matrix[:, t, :, :] = torch.matmul(x[:, t, :, :], mat.T)

# ==========================================
# 验证结果
# ==========================================
diff = (out_vector - out_matrix).abs().max()

print("-" * 30)
print(f"最大误差 (float64): {diff.item():.25f}")
print("-" * 30)

if diff < 1e-14:
    print("✅ 验证成功：误差已降至 1e-15 级别，这才是 float64 应有的表现！")
    print("   证明：之前的误差完全来源于 float32 的精度限制。")
else:
    print("❌ 依然存在误差，请检查是否还有隐式类型转换。")

当前精度模式: torch.float64
------------------------------
最大误差 (float64): 0.0000000000000008881784197
------------------------------
✅ 验证成功：误差已降至 1e-15 级别，这才是 float64 应有的表现！
   证明：之前的误差完全来源于 float32 的精度限制。


In [ ]:
# 参数配置
batch_size = 1
seq_len = 8   
heads = 1      
head_dim = 128
theta = 10000.0

# 构造输入 [bs, seq, heads, dim]
# 使用 repeat 构造完全相同的向量，控制变量
q_vec = torch.randn(batch_size, 1, heads, head_dim)
k_vec = torch.randn(batch_size, 1, heads, head_dim)
q = q_vec.repeat(1, seq_len, 1, 1)
k = k_vec.repeat(1, seq_len, 1, 1)

# 应用 RoPE (在 [bs, seq, heads, dim] 维度下进行，符合你的函数设计)
freqs = compute_freqs(head_dim, seq_len, theta)
q_rope = apply_rope_element(q, freqs)
k_rope = apply_rope_element(k, freqs)

# 维度置换 (Permute/Transpose)
# 变换前: [bs, seq, heads, dim]
# 变换后: [bs, heads, seq, dim]
q_rope = q_rope.transpose(1, 2)
k_rope = k_rope.transpose(1, 2)

# 计算 Attention Matrix: [bs, heads, seq, dim] @ [bs, heads, dim, seq] -> [bs, heads, seq, seq]
attention_rope = torch.matmul(q_rope, k_rope.transpose(-2, -1))

print(f"attention_rope.shape: {attention_rope.shape}")

# 提取矩阵 [seq, seq]
attn_matrix = attention_rope[0, 0]
attn_matrix